In [ ]:
import sys
import os

#adding parent directory of this notebook to python path to import modules
#it might not work if we try to import modules then
#os.getcwd() --gives the path of the current directory D:\Generative AI Dev Projects\RAG_Capstone_Project\notebooks
#os.path.dirname(os.getcwd())) --Projects\RAG_Capstone_Project
#sys.path is the list of directories Python searches through when you write import something,now this will include the parent directory also
    
sys.path.append(os.path.dirname(os.getcwd()))

from modules.loader import load_document
from modules.splitter import split_documents

import warnings
warnings.filterwarnings("ignore")


In [ ]:
sample_txt = "../data/sample_docs/sample_text.txt"
if not os.path.exists(sample_txt):
    os.makedirs(os.path.dirname(sample_txt),exist_ok=True)
    with open(sample_txt,"w",encoding ="utf-8") as f:
        f.write("This is a short sample document abouit Langchain \n * 50")

print("====Loading TXT ====")
docs = load_document(sample_txt)


print("Loaded documents:",len(docs))
print("First 50 chars:" , docs[0].page_content[:50])







In [ ]:
print("=== Splitting into chunks ===")
chunks = split_documents(docs,chunk_size = 200, chunk_overlap = 50)
print("Total Chunks : " ,len(chunks))
print(f"Chunk 0 (first 200 chars): \n {chunks[0].page_content[:200]}")
print(f"Chunk 1 (first 200 chars): \n {chunks[1].page_content[:200]}")
print(f"Chunk 0 metadata : {chunks[0].metadata}")

In [ ]:
from modules.utils import get_embedding_model

for provider in ["openai","gemini"]:
    try:
        print(f"=== Testing {provider.upper()} ===")
        model = get_embedding_model(provider)
        print("Model loaded: ",type(model))
    except Exception as e:
        print(e)

In [ ]:
from modules.embedder import build_or_update_vectorstore
import json

#load config 
with open("../config/config.json","r") as f:
    config = json.load(f)


# ---- added: check which file was read and what it says ----
print("Config file path:", os.path.abspath("../config/config.json"))
print("Vector store type in memory:", config["vector_store"]["type"])

#load and split documents


sample_txt = "../data/sample_docs/sample_text.txt"
docs = load_document(sample_txt)
chunks = split_documents(docs,chunk_size = 200,chunk_overlap = 40)
















In [ ]:
#vectorstore = build_or_update_vectorstore(chunks,config)
#print(vectorstore.index.ntotal)


import os
from pinecone import Pinecone

vectorstore = build_or_update_vectorstore(chunks, config)

pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
print(pc.Index("rag-assistant-index").describe_index_stats())

In [ ]:
"""#Test a similarity search
query = "How many stages RAG pipeline has? explain each in one sentence"
results = vectorstore.similarity_search(query,k=2)
print(f"Query: {query} \n\n ")

for i,doc in enumerate(results,1):
    print(f"{i} : {doc.page_content[:100]}\n\n")

"""






















In [ ]:


from modules.retriever import get_retriever

retriever = get_retriever(vectorstore,config) #either base or multiquery


"""
query = " Explain RAG stages in short"

print(f"Query : {query}\n")

results = retriever.invoke(query)

for i,doc in enumerate(results,1):
    print(f"{i}: {doc.page_content[:100]}\n\n")

"""


docs = retriever.invoke("Whose pencil broke?")
print(len(docs))
for d in docs:
    print(d.page_content[:150], "\n---")



In [ ]:
"""

from modules.rag_chain import build_rag_chain
from modules.memory import add_memory_to_chain
from modules.memory import get_session_summary_memory, get_memory_summary

session_id = "test_user_001"

#use the rag chain
rag_chain = build_rag_chain(retriever,config)


rag_with_memory = add_memory_to_chain(rag_chain,session_id,enabled = True)

print("\n First question : ")
resp1 = rag_with_memory.invoke(
    {"question" : "What are the friends' names? and Where was the missing pencil found?"},
    config = {"configurable": {"session_id" : session_id}},
)

print(f"\n{resp1.content}")

#claude-begin
mem = get_session_summary_memory("test_user_001")
print(repr(mem.buffer))                       # the real summary string
print(mem.load_memory_variables({}))          # {'history': [SystemMessage(...)]}
print(get_memory_summary("test_user_001"))  

#claude - end  # currently prints an empty line
print("\n Followup question : ")
resp2 = rag_with_memory.invoke(
    {"question" : "Summarize your previous response in 3 bullet points"},
    config = {"configurable": {"session_id" : session_id}},
)

print(f"\n{resp2.content}")

"""

In [ ]:
from modules.rag_chain import build_rag_chain
from modules.memory import add_memory_to_chain
from modules.memory import get_session_summary_memory, get_memory_summary

session_id = "test_user_001"

rag_chain = build_rag_chain(retriever, config)
rag_with_memory = add_memory_to_chain(rag_chain, session_id, enabled=True)


while True:
    q = input("You: ")
    if q.lower() in ("quit", "exit"):
        break
    resp = rag_with_memory.invoke(
        {"question": q},
        config={"configurable": {"session_id": session_id}},
    )
    print("Bot:", resp.content, "\n")